# 2023 Kahramanmaraş M7.8 — ShakeMap Comparison

On February 6, 2023, a M7.8 earthquake struck the East Anatolian Fault zone,
causing over 50,000 fatalities across southern Turkey and Syria.

This notebook demonstrates how different inputs progressively improve ShakeMap:

| Step | Finite fault | Station data | What we learn |
|------|-------------|--------------|---------------|
| 2 | ✓ | — | Fault geometry controls shaking pattern |
| 3 | — | — | Point source misses near-fault directivity |
| 4 | — | Predicted (from FF) | Dense data improves even a point source |
| 5 | — | Real observations | Real data best matches what happened |

## 1. Setup

In [ ]:
import os, json, shutil, glob
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd

EVENT = 'us6000jllz'
os.environ['EVENT'] = EVENT

REPO      = Path('/workspaces/shakemap-gf-workshop')
EVENT_CFG = REPO / 'data/event_configs' / EVENT
EVENT_DIR = Path.home() / 'shakemap_profiles/default/data' / EVENT / 'current'
DATA_DIR  = EVENT_DIR / 'data'
PRODUCTS  = EVENT_DIR / 'products'

print(f'Event: {EVENT}  (M7.8 Kahramanmaraş, 2023-02-06)')
print(f'Repo config: {EVENT_CFG.exists()}')

## 2. Stage the event

In [ ]:
%%bash
sm_create $EVENT

In [ ]:
shutil.copy(EVENT_CFG / 'event.xml',    DATA_DIR / 'event.xml')
shutil.copy(EVENT_CFG / 'rupture.json', DATA_DIR / 'rupture.json')
shutil.copy(EVENT_CFG / 'model.conf',   EVENT_DIR / 'model.conf')
print('Event files staged')
print(f'Rupture: {(DATA_DIR / "rupture.json").exists()}')

## 3. Step 2: Grid mode with finite fault

Run ShakeMap using the finite fault rupture geometry but no station observations.
This shows what the GMPE predicts using only the fault model.

In [ ]:
%%bash
shake $EVENT select assemble -c 'grid FF no stations' model contour mapping gridxml \
  2>&1 | grep -E 'Running|Finished|ERROR|WARNING'

In [ ]:
shutil.copy(PRODUCTS / 'intensity.jpg', Path.home() / 'step2_FF_grid.jpg')
plt.figure(figsize=(10, 8))
plt.imshow(mpimg.imread(PRODUCTS / 'intensity.jpg'))
plt.title('Step 2: Grid mode — finite fault, no stations', fontsize=13)
plt.axis('off')
plt.tight_layout()
plt.show()

## 4. Step 3: Grid mode with point source

Remove the rupture.json so ShakeMap uses a point source instead of the finite fault.

In [ ]:
rupture = DATA_DIR / 'rupture.json'
rupture.rename(DATA_DIR / 'rupture.json.bak')
print('rupture.json removed — ShakeMap will use point source')

In [ ]:
%%bash
shake $EVENT select assemble -c 'grid point source' model contour mapping gridxml \
  2>&1 | grep -E 'Running|Finished|ERROR|WARNING'

In [ ]:
shutil.copy(PRODUCTS / 'intensity.jpg', Path.home() / 'step3_pointsource_grid.jpg')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, path, title in [
    (axes[0], Path.home() / 'step2_FF_grid.jpg',         'Step 2: Finite fault'),
    (axes[1], Path.home() / 'step3_pointsource_grid.jpg', 'Step 3: Point source'),
]:
    ax.imshow(mpimg.imread(path))
    ax.set_title(title, fontsize=12)
    ax.axis('off')
plt.suptitle('Effect of finite fault vs point source — M7.8 Kahramanmaraş', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Step 1: Points mode — predict shaking at station locations

Run ShakeMap in points mode using the finite fault, predicting shaking at the
locations of the 262 real stations. This gives us a synthetic dataset
representing what the finite fault model predicts at each station.

In [ ]:
# Restore finite fault
(DATA_DIR / 'rupture.json.bak').rename(DATA_DIR / 'rupture.json')

# Extract station locations from stationlist.json → sites.csv
with open(EVENT_CFG / 'stationlist.json') as f:
    stations = json.load(f)

rows = []
for feat in stations['features']:
    props = feat['properties']
    lon, lat = feat['geometry']['coordinates'][:2]
    rows.append({
        'id':  props.get('code', ''),
        'lat': lat,
        'lon': lon,
        'vs30': props.get('vs30', 760.0) or 760.0
    })

sites_df = pd.DataFrame(rows)
sites_csv = DATA_DIR / 'sites.csv'
sites_df.to_csv(sites_csv, index=False)
print(f'Wrote {len(sites_df)} station locations to sites.csv')

In [ ]:
# Add sites.csv to model.conf for points mode
model_conf_path = EVENT_DIR / 'model.conf'
conf_text = model_conf_path.read_text()
if 'prediction_location' not in conf_text:
    conf_text += f"""
[interp]
    [[prediction_location]]
        file = {sites_csv}
"""
    model_conf_path.write_text(conf_text)
print('model.conf updated for points mode')

In [ ]:
%%bash
shake $EVENT select assemble -p -c 'points mode FF' model makecsv -g \
  2>&1 | grep -E 'Running|Finished|ERROR|WARNING'

In [ ]:
synthetic_station_files = glob.glob(str(PRODUCTS / '*points*.json'))
print('Generated files:', synthetic_station_files)

## 6. Step 4: Grid mode — point source + synthetic stations

Use the synthetic station data from the finite fault points run as input to a
point source grid run. This shows how much station density alone can improve
a point source model.

In [ ]:
# Remove rupture (point source) and copy synthetic stations as input
(DATA_DIR / 'rupture.json').rename(DATA_DIR / 'rupture.json.bak')
if synthetic_station_files:
    shutil.copy(synthetic_station_files[0], DATA_DIR / 'stationlist.json')
    print('Synthetic stations copied as input')

# Remove points mode from model.conf
conf_text = model_conf_path.read_text()
conf_text = '\n'.join([l for l in conf_text.splitlines()
                       if 'prediction_location' not in l and 'sites.csv' not in l])
model_conf_path.write_text(conf_text)
print('model.conf restored to grid mode')

In [ ]:
%%bash
shake $EVENT select assemble -c 'grid point source + FF stations' model contour mapping gridxml \
  2>&1 | grep -E 'Running|Finished|ERROR|WARNING'

In [ ]:
shutil.copy(PRODUCTS / 'intensity.jpg', Path.home() / 'step4_ps_ffstations.jpg')

fig, axes = plt.subplots(1, 3, figsize=(20, 7))
for ax, path, title in [
    (axes[0], Path.home() / 'step2_FF_grid.jpg',         'Step 2: Finite fault'),
    (axes[1], Path.home() / 'step3_pointsource_grid.jpg', 'Step 3: Point source'),
    (axes[2], Path.home() / 'step4_ps_ffstations.jpg',   'Step 4: Point source\n+ FF-predicted stations'),
]:
    ax.imshow(mpimg.imread(path))
    ax.set_title(title, fontsize=11)
    ax.axis('off')
plt.suptitle('Progressive improvement — M7.8 Kahramanmaraş', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Step 5: Grid mode with real station data

Now use the actual 262 station observations. This matches what appears on
earthquake.usgs.gov — the operational ShakeMap product.

In [ ]:
# Restore finite fault and use real stations
(DATA_DIR / 'rupture.json.bak').rename(DATA_DIR / 'rupture.json')
shutil.copy(EVENT_CFG / 'stationlist.json', DATA_DIR / 'stationlist.json')
print('Real stationlist.json and rupture.json restored')

In [ ]:
%%bash
shake $EVENT select assemble -c 'grid real stations' model contour mapping gridxml \
  2>&1 | grep -E 'Running|Finished|ERROR|WARNING'

In [ ]:
shutil.copy(PRODUCTS / 'intensity.jpg', Path.home() / 'step5_real_stations.jpg')

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
for ax, path, title in [
    (axes[0,0], Path.home() / 'step2_FF_grid.jpg',         'Step 2: Finite fault, no data'),
    (axes[0,1], Path.home() / 'step3_pointsource_grid.jpg', 'Step 3: Point source, no data'),
    (axes[1,0], Path.home() / 'step4_ps_ffstations.jpg',   'Step 4: Point source\n+ FF-predicted stations'),
    (axes[1,1], Path.home() / 'step5_real_stations.jpg',   'Step 5: Finite fault\n+ real observations'),
]:
    ax.imshow(mpimg.imread(path))
    ax.set_title(title, fontsize=11)
    ax.axis('off')
plt.suptitle('ShakeMap improvement with progressive data — M7.8 Kahramanmaraş', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Discussion

- **Steps 2 vs 3:** How does the finite fault change the spatial pattern of shaking?
  Where does the elongation of shaking along the fault make the biggest difference?
- **Steps 3 vs 4:** Can dense predicted station data compensate for a poor source model?
- **Steps 4 vs 5:** Where do the real observations change the ShakeMap most?
  Are there areas where the model was consistently over- or under-predicting?
- **Overall:** What does this tell us about the relative importance of fault geometry
  vs station data for operational ShakeMap?

---
## 9. Exercise: What if only part of the fault ruptured?

The M7.8 rupture propagated ~350 km along the East Anatolian Fault.
What if only part of the fault had ruptured — how would shaking in Gaziantep
and Adıyaman differ?

Modify `LAT_THRESHOLD` below to control which part of the fault ruptures,
then rerun ShakeMap and compare.

In [ ]:
import copy

with open(EVENT_CFG / 'rupture.json') as f:
    rupture_full = json.load(f)

coords = rupture_full['features'][0]['geometry']['coordinates']
print(f'Full rupture: {len(coords)} polygons')
for i, poly in enumerate(coords):
    pts = poly[0]
    top = [p for p in pts if p[2] == 0.0]
    if top:
        lats = [p[1] for p in top]
        lons = [p[0] for p in top]
        print(f'  Polygon {i+1}: lat {min(lats):.2f}–{max(lats):.2f}, lon {min(lons):.2f}–{max(lons):.2f}')

In [ ]:
# Keep only polygons with mean latitude below this threshold
# Try: 37.5 (southern half), 37.0 (southern third), 38.0 (most of fault)
LAT_THRESHOLD = 37.5

rupture_partial = copy.deepcopy(rupture_full)
original_polys = rupture_partial['features'][0]['geometry']['coordinates']
filtered_polys = []
for poly in original_polys:
    pts = poly[0]
    top = [p for p in pts if p[2] == 0.0]
    if top:
        mean_lat = sum(p[1] for p in top) / len(top)
        if mean_lat < LAT_THRESHOLD:
            filtered_polys.append(poly)

rupture_partial['features'][0]['geometry']['coordinates'] = filtered_polys
print(f'Original polygons: {len(original_polys)}')
print(f'Filtered (lat < {LAT_THRESHOLD}°N): {len(filtered_polys)}')

with open(DATA_DIR / 'rupture.json', 'w') as f:
    json.dump(rupture_partial, f)
print('Partial rupture.json written')

In [ ]:
%%bash
shake $EVENT select assemble -c 'partial rupture' model contour mapping gridxml \
  2>&1 | grep -E 'Running|Finished|ERROR|WARNING'

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, path, title in [
    (axes[0], Path.home() / 'step5_real_stations.jpg', 'Full rupture (~350 km)'),
    (axes[1], PRODUCTS / 'intensity.jpg',              f'Partial rupture (lat < {LAT_THRESHOLD}°N)'),
]:
    ax.imshow(mpimg.imread(str(path)))
    ax.set_title(title, fontsize=12)
    ax.axis('off')
plt.suptitle('Effect of rupture length — M7.8 Kahramanmaraş', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Restore full rupture
shutil.copy(EVENT_CFG / 'rupture.json', DATA_DIR / 'rupture.json')
print('Full rupture.json restored')

**Discussion:**
- Which cities lose significant shaking when only part of the fault ruptures?
- How does the rupture length affect the area of MMI ≥ VII shaking?
- Try adjusting `LAT_THRESHOLD` to explore different rupture extents.
  What threshold roughly halves the affected area?